# Remote Cleanup Interactive Manual Tests

This notebook is intentionally simple and repetitive. There are no notebook helper functions: most error cells show the exact normal PYNQ call being tested, and the final section directly calls raw RPC stubs for malformed-handle edge cases.

Watch the `pynq-remote` logs on the board while you run the cells. Many cells are supposed to raise or print gRPC errors. Those errors are the success condition.

When a section says to re-run START 1 or START 2, re-run those cells to recreate the device/resources needed for that specific error test.

## Setup

Run this once after opening the notebook and again after any kernel restart.

In [4]:
import gc
import json
import os
from pathlib import Path

import numpy as np

REMOTE_IP = os.environ.get("PYNQ_REMOTE_DEVICES", "192.168.2.197").split(",")[0].strip()
os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP
OVERLAY_PATH = "/workspace/phd/PYNQ.remote-dev/applications/PYNQ/tests/resizer.xsa"

import pynq

print("REMOTE_IP:", REMOTE_IP)
print("OVERLAY_PATH:", OVERLAY_PATH)

REMOTE_IP: 192.168.2.197
OVERLAY_PATH: /workspace/phd/PYNQ.remote-dev/applications/PYNQ/tests/resizer.xsa


## START 1: Create A Device

Use the normal PYNQ probe cell for single-client behavior. Use the no-cleanup alternate only for tests that explicitly say so.

In [5]:
# START 1: create a RemoteDevice through normal PYNQ import/probe.
# Re-run this cell when you want constructor auto_cleanup=True to run again.
if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")

remote_devices = [d for d in pynq.Device.devices if d.has_capability("REMOTE")]
if not remote_devices:
    raise RuntimeError("No RemoteDevice found. Check PYNQ_REMOTE_DEVICES and board connectivity.")

device = remote_devices[0]
print("device:", device)

device: <pynq.pl_server.remote_device.RemoteDevice object at 0x73792ced4350>


In [ ]:
# START 1 ALT: create a RemoteDevice without constructor auto-cleanup.
# Use this only when a test explicitly needs to leave existing target resources alone.
from pynq.pl_server.remote_device import RemoteDevice
device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=False)
print("manual no-cleanup device:", device)

## START 2: Create Test Resources

Run START 1 first, then run this cell. Re-run this cell whenever a later error cell has released or cleaned the current resources.

In [6]:
# START 2: load the overlay and create one MMIO, one buffer, and optional GPIO.
# Re-run START 1 first if you want a fresh device/constructor cleanup.
overlay = pynq.Overlay(OVERLAY_PATH, device=device)
resizer = overlay.resize_accel_0
resizer.read(0)

mmio = resizer.mmio
mmio_id = mmio._remote_map.mmio_id

buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
buffer[:] = np.arange(16, dtype=np.uint32)
buffer.flush()
buffer_id = buffer.buffer_id

gpio = None
gpio_id = None
gpio_pin = None
gpio_path = None
base_path = pynq.GPIO.get_gpio_base_path(device=device)
npins = pynq.GPIO.get_gpio_npins(device=device)
if base_path and npins:
    gpio_pin = pynq.GPIO.get_gpio_pin(0, device=device)
    gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if device.exists_file(gpio_path).exists:
        print("GPIO already exported, skipping GPIO object:", gpio_path)
    else:
        gpio = pynq.GPIO(gpio_pin, "in", device=device)
        gpio.read()
        gpio_id = gpio._gpio_id
else:
    print("GPIO sysfs base not available; GPIO cells will skip.")

print("MMIO id:  ", mmio_id)
print("Buffer id:", buffer_id)
print("GPIO id:  ", gpio_id, "path:", gpio_path)

MMIO id:   6:19
Buffer id: 6:0
GPIO id:   6:0 path: /sys/class/gpio/gpio594


## Normal PYNQ API Error Cells

Malformed remote IDs are not normally reachable through the public PYNQ API, because PYNQ receives opaque IDs from the server. This section uses normal PYNQ operations that should fail visibly. The final section imports the `pb2` modules and calls raw RPC stubs for the malformed-ID boundary checks.

## Individual Release Errors

Run START 2 before each of these cells if you want to test each service independently. Each cell releases one live PYNQ object, then immediately uses that same normal PYNQ object again. The cell should error. For exact server-side `Object not found` assertions, use the automated pytest tests.

In [7]:
# Error test: release one current MMIO, then use that same normal PYNQ object again.
# Re-run START 2 before this cell if mmio was already released or cleaned.
mmio.close()
mmio.read(0)

RuntimeError: Remote MMIO handle has been closed or invalidated.

In [8]:
# Error test: free one current buffer, then use that same normal PYNQ object again.
# Re-run START 2 before this cell if buffer was already released or cleaned.
buffer.freebuffer()
buffer.physical_address

RuntimeError: Remote buffer has been freed or invalidated.

In [9]:
# Error test: release one current GPIO, then use that same normal PYNQ object again.
# Re-run START 2 before this cell if gpio was already released or cleaned.
if gpio_id is None:
    print("Skipping GPIO release test because no GPIO object was created.")
else:
    gpio.release()
    gpio.read()

RuntimeError: Remote GPIO handle has been released or invalidated.

## Global Cleanup Stale Errors

Run START 2, then the cleanup trigger. The following MMIO, Buffer, and GPIO cells each use the old normal PYNQ object after global cleanup. These cells should error, usually with gRPC `NOT_FOUND` / `Object stale` details from the server.

In [ ]:
# Cleanup trigger: run START 2 first, then this cell.
print("old MMIO id:  ", mmio_id)
print("old Buffer id:", buffer_id)
print("old GPIO id:  ", gpio_id)
response = device.cleanup()
print(response)
print("You should see Cleanup Request Received in the target logs.")

In [ ]:
# Error test: after global cleanup, using the old normal PYNQ MMIO object should fail.
mmio.read(0)

In [ ]:
# Error test: after global cleanup, using the old normal PYNQ Buffer object should fail.
buffer.physical_address

In [ ]:
# Error test: after global cleanup, using the old normal PYNQ GPIO object should fail.
if gpio_id is None:
    print("Skipping GPIO stale test because no GPIO object was created.")
else:
    gpio.read()

In [ ]:
# Release-style normal PYNQ cleanup calls should be safe even after global cleanup.
# Run this after the cleanup trigger. It should not error.
mmio.close()
buffer.freebuffer()
if gpio is not None:
    gpio.release()
print("Release-style PYNQ cleanup calls completed.")

In [10]:
# Fresh allocation after cleanup should not alias the old handles.
fresh_overlay = pynq.Overlay(OVERLAY_PATH, device=device)
fresh_resizer = fresh_overlay.resize_accel_0
fresh_resizer.read(0)
fresh_mmio = fresh_resizer.mmio
fresh_mmio_id = fresh_mmio._remote_map.mmio_id

fresh_buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
fresh_buffer_id = fresh_buffer.buffer_id

print("old -> fresh MMIO:  ", mmio_id, "->", fresh_mmio_id)
print("old -> fresh Buffer:", buffer_id, "->", fresh_buffer_id)
assert fresh_mmio_id != mmio_id
assert fresh_buffer_id != buffer_id
print("Fresh handles did not alias old handles.")

old -> fresh MMIO:   6:19 -> 7:19
old -> fresh Buffer: 6:0 -> 7:0
Fresh handles did not alias old handles.


## Full Overlay Download Cleanup

Run START 2, then this trigger cell. A full `Overlay(...)` load should cleanup old resources before new overlay setup. Then run the stale MMIO/Buffer/GPIO cells above.

In [11]:
# Overlay cleanup trigger: run START 2 first, then this cell.
print("old MMIO id before second overlay:  ", mmio_id)
print("old Buffer id before second overlay:", buffer_id)
print("old GPIO id before second overlay:  ", gpio_id)
overlay_after_cleanup = pynq.Overlay(OVERLAY_PATH, device=device)
print("Second full overlay load completed. You should see Cleanup Request Received before new overlay setup.")

old MMIO id before second overlay:   6:19
old Buffer id before second overlay: 6:0
old GPIO id before second overlay:   6:0
Second full overlay load completed. You should see Cleanup Request Received before new overlay setup.


## Constructor Auto-Cleanup Across Kernel Restart

Run the seed cell, restart the kernel, then run Setup and the verification cell. This tests the single-client rule that a new auto-cleanup device construction clears resources left by an earlier client.

In [12]:
# Cross-kernel setup: create orphan-prone resources without constructor auto-cleanup.
# Run Setup first. Run this cell, check the target logs, then restart the kernel.
from pynq.pl_server.remote_device import RemoteDevice
seed_device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=False)
seed_overlay = pynq.Overlay(OVERLAY_PATH, device=seed_device)
seed_resizer = seed_overlay.resize_accel_0
seed_resizer.read(0)
seed_mmio = seed_resizer.mmio
seed_buffer = seed_device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
seed_buffer[:] = np.arange(16, dtype=np.uint32)
seed_buffer.flush()

seed_gpio = None
seed_gpio_id = None
seed_gpio_path = None
base_path = pynq.GPIO.get_gpio_base_path(device=seed_device)
npins = pynq.GPIO.get_gpio_npins(device=seed_device)
if base_path and npins:
    seed_gpio_pin = pynq.GPIO.get_gpio_pin(0, device=seed_device)
    seed_gpio_path = f"/sys/class/gpio/gpio{seed_gpio_pin}"
    if not seed_device.exists_file(seed_gpio_path).exists:
        seed_gpio = pynq.GPIO(seed_gpio_pin, "in", device=seed_device)
        seed_gpio.read()
        seed_gpio_id = seed_gpio._gpio_id

STATE_FILE = Path("/tmp/pynq_remote_cleanup_constructor_state.json")
state = {
    "mmio_id": seed_mmio._remote_map.mmio_id,
    "buffer_id": seed_buffer.buffer_id,
    "gpio_id": seed_gpio_id,
    "gpio_path": seed_gpio_path,
}
STATE_FILE.write_text(json.dumps(state, indent=2))
print(state)
print("Restart the kernel now. After restart, run Setup and then the constructor verification cell.")

{'mmio_id': '8:38', 'buffer_id': '8:0', 'gpio_id': '8:0', 'gpio_path': '/sys/class/gpio/gpio594'}
Restart the kernel now. After restart, run Setup and then the constructor verification cell.


In [13]:
# Constructor auto-cleanup verification using normal PYNQ discovery.
# Run Setup first after the restart, then this cell.
STATE_FILE = Path("/tmp/pynq_remote_cleanup_constructor_state.json")
state = json.loads(STATE_FILE.read_text())
print("old handles saved before restart:", state)

# Normal PYNQ probing constructs a RemoteDevice with auto_cleanup=True.
if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")
cleanup_device = [d for d in pynq.Device.devices if d.has_capability("REMOTE")][0]
print("constructed auto-cleanup device:", cleanup_device)
print("Check target logs: constructor cleanup should have run before any fresh resources below.")

fresh_overlay = pynq.Overlay(OVERLAY_PATH, device=cleanup_device)
fresh_resizer = fresh_overlay.resize_accel_0
print("fresh register read:", fresh_resizer.read(0))
fresh_buffer = cleanup_device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
fresh_buffer[:] = np.arange(16, dtype=np.uint32)
fresh_buffer.flush()
print("fresh buffer physical address:", fresh_buffer.physical_address)
fresh_buffer.freebuffer()

old handles saved before restart: {'mmio_id': '8:38', 'buffer_id': '8:0', 'gpio_id': '8:0', 'gpio_path': '/sys/class/gpio/gpio594'}
constructed auto-cleanup device: <pynq.pl_server.remote_device.RemoteDevice object at 0x73791ebe2050>
Check target logs: constructor cleanup should have run before any fresh resources below.
fresh register read: 4
fresh buffer physical address: 34371776512


## Atexit Cleanup Across Kernel Restart

Run the seed cell, restart the kernel, then run Setup and the verification cell. If the old handles already fail after restart, orderly kernel shutdown ran `atexit` cleanup.

In [14]:
# Atexit setup: create live resources through normal PYNQ probing, then restart the kernel.
# Do not manually call cleanup or release. The point is to see whether orderly kernel restart runs atexit cleanup.
if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")
atexit_device = [d for d in pynq.Device.devices if d.has_capability("REMOTE")][0]
atexit_overlay = pynq.Overlay(OVERLAY_PATH, device=atexit_device)
atexit_resizer = atexit_overlay.resize_accel_0
atexit_resizer.read(0)
atexit_mmio = atexit_resizer.mmio
atexit_buffer = atexit_device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
atexit_buffer[:] = np.arange(16, dtype=np.uint32)
atexit_buffer.flush()

atexit_gpio = None
atexit_gpio_id = None
atexit_gpio_path = None
base_path = pynq.GPIO.get_gpio_base_path(device=atexit_device)
npins = pynq.GPIO.get_gpio_npins(device=atexit_device)
if base_path and npins:
    atexit_gpio_pin = pynq.GPIO.get_gpio_pin(0, device=atexit_device)
    atexit_gpio_path = f"/sys/class/gpio/gpio{atexit_gpio_pin}"
    if not atexit_device.exists_file(atexit_gpio_path).exists:
        atexit_gpio = pynq.GPIO(atexit_gpio_pin, "in", device=atexit_device)
        atexit_gpio.read()
        atexit_gpio_id = atexit_gpio._gpio_id

STATE_FILE = Path("/tmp/pynq_remote_cleanup_atexit_state.json")
state = {
    "mmio_id": atexit_mmio._remote_map.mmio_id,
    "buffer_id": atexit_buffer.buffer_id,
    "gpio_id": atexit_gpio_id,
    "gpio_path": atexit_gpio_path,
}
STATE_FILE.write_text(json.dumps(state, indent=2))
print(state)
print("Restart the kernel now. Watch for the final Cleanup Request in the target logs.")

{'mmio_id': '10:19', 'buffer_id': '10:0', 'gpio_id': '10:0', 'gpio_path': '/sys/class/gpio/gpio594'}
Restart the kernel now. Watch for the final Cleanup Request in the target logs.


In [15]:
# Atexit verification using normal PYNQ discovery.
# Run Setup first after restart, then this cell.
STATE_FILE = Path("/tmp/pynq_remote_cleanup_atexit_state.json")
state = json.loads(STATE_FILE.read_text())
print("old handles saved before restart:", state)
print("Check the target logs for a Cleanup Request when the previous kernel restarted.")

if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")
device = [d for d in pynq.Device.devices if d.has_capability("REMOTE")][0]
print("fresh auto-cleanup device:", device)
print("If atexit did not clean the old resources, this constructor cleanup should clean them now.")

overlay = pynq.Overlay(OVERLAY_PATH, device=device)
resizer = overlay.resize_accel_0
print("fresh register read:", resizer.read(0))

old handles saved before restart: {'mmio_id': '10:19', 'buffer_id': '10:0', 'gpio_id': '10:0', 'gpio_path': '/sys/class/gpio/gpio594'}
Check the target logs for a Cleanup Request when the previous kernel restarted.
fresh auto-cleanup device: <pynq.pl_server.remote_device.RemoteDevice object at 0x73791e46c4d0>
If atexit did not clean the old resources, this constructor cleanup should clean them now.
fresh register read: 4


## Non-API Raw RPC Edge Cases

Run Setup or START 1 first so `device` exists. These cells intentionally bypass the public PYNQ API and send malformed handles directly to the gRPC server. Each operation should print `INVALID_ARGUMENT` and an `Invalid ... ID` message. This is useful for checking the server boundary, but it is not normal user-facing PYNQ usage.

In [16]:
import grpc
from pynq.remote import buffer_pb2, gpio_pb2, mmio_pb2

if "device" not in globals():
    remote_devices = [d for d in pynq.Device.devices if d.has_capability("REMOTE")]
    if not remote_devices:
        raise RuntimeError("No RemoteDevice found. Check PYNQ_REMOTE_DEVICES and board connectivity.")
    device = remote_devices[0]

print("raw RPC checks will use device:", device)

raw RPC checks will use device: <pynq.pl_server.remote_device.RemoteDevice object at 0x73791e46c4d0>


In [17]:
# Non-API error test: malformed MMIO handle.
try:
    device._stub["mmio"].read(
        mmio_pb2.ReadRequest(
            mmio_id="not-a-valid-mmio-id",
            offset=0,
            length=4,
            word_order="little",
        )
    )
except grpc.RpcError as exc:
    print(exc.code().name, exc.details())
    assert exc.code() == grpc.StatusCode.INVALID_ARGUMENT
    assert "Invalid MMIO ID" in exc.details()
else:
    raise AssertionError("Malformed MMIO handle unexpectedly succeeded")

INVALID_ARGUMENT Invalid MMIO ID while handling read() for mmio_id='not-a-valid-mmio-id'.


In [18]:
# Non-API error test: malformed Buffer handle.
try:
    device._stub["buffer"].physical_address(
        buffer_pb2.AddressRequest(buffer_id="not-a-valid-buffer-id")
    )
except grpc.RpcError as exc:
    print(exc.code().name, exc.details())
    assert exc.code() == grpc.StatusCode.INVALID_ARGUMENT
    assert "Invalid Buffer ID" in exc.details()
else:
    raise AssertionError("Malformed Buffer handle unexpectedly succeeded")

INVALID_ARGUMENT Invalid Buffer ID while handling physical_address() for buffer_id='not-a-valid-buffer-id'.


In [19]:
# Non-API error test: malformed GPIO handle.
try:
    device._stub["gpio"].read(gpio_pb2.GpioReadRequest(gpio_id="not-a-valid-gpio-id"))
except grpc.RpcError as exc:
    print(exc.code().name, exc.details())
    assert exc.code() == grpc.StatusCode.INVALID_ARGUMENT
    assert "Invalid GPIO ID" in exc.details()
else:
    raise AssertionError("Malformed GPIO handle unexpectedly succeeded")

INVALID_ARGUMENT Invalid GPIO ID while handling read() for gpio_id='not-a-valid-gpio-id'.
